# ST-Reason · 00 Setup and shared utilities

**What this notebook does.** It mounts Google Drive, creates the project folders, writes `config/config.yaml` the first time, and defines the helper functions every other notebook uses: resumable loops, safe JSON/JSONL writes, the run manifest, memory monitoring and video staging.

**How to use it.**
1. Run it once on its own (Runtime → Run all). It sets up the folders and prints a system report. No GPU is needed.
2. Every other notebook loads it automatically with `%run -i` in its first cell, so the helpers are always identical.

**Where things go.** Everything is saved under `MyDrive/ST-Reason/`. Nothing important is kept only in the Colab VM.

In [1]:
# --- 1. Environment and Google Drive -------------------------------------------------
import os, sys, json, time, csv, gc, re, shutil, hashlib, random, zipfile, subprocess, platform, threading
import datetime as dt
from pathlib import Path
import importlib.util

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')
RUNTIME = 'colab' if IN_COLAB else ('kaggle' if IN_KAGGLE else 'local')
PROJECT_NAME = 'ST-Reason'

if IN_COLAB:
    if not os.path.ismount('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive') / PROJECT_NAME      # persistent (Drive)
    LOCAL = Path('/content/st_local')                        # fast, wiped when the VM resets
elif IN_KAGGLE:
    BASE = Path('/kaggle/working') / PROJECT_NAME            # zipped and moved to Drive afterwards
    LOCAL = Path('/tmp/st_local')
else:                                                        # local testing
    BASE = Path(os.environ.get('ST_REASON_BASE', Path.cwd() / 'ST-Reason_local')).resolve()
    LOCAL = Path(os.environ.get('ST_REASON_LOCAL', Path.cwd() / 'st_local')).resolve()

ST_NOTEBOOK = globals().get('ST_NOTEBOOK', '00_setup_and_utils')
print(f'Runtime: {RUNTIME} | project folder: {BASE} | local scratch: {LOCAL}')

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local


In [2]:
# --- 2. Folder layout (mirrors the plan's Drive layout) ------------------------------------
P = {
    'config':   BASE / 'config',
    'prompts':  BASE / 'prompts',
    'notebooks': BASE / 'notebooks',
    'data':     BASE / 'data',
    'ann':      BASE / 'data' / 'annotations',
    'subsets':  BASE / 'data' / 'subsets',
    'source':   BASE / 'data' / 'source',      # put the Drive shortcut to NExTVideo.zip here
    'stgap':    BASE / 'data' / 'stgap',
    'inbox':    BASE / 'inbox',                # zips coming back from Kaggle runs
    'outputs':  BASE / 'outputs',
    'cv':       BASE / 'outputs' / 'cv',
    'frames':   BASE / 'outputs' / 'frames',
    'memory':   BASE / 'outputs' / 'memory',
    'pred':     BASE / 'outputs' / 'predictions',
    'metrics':  BASE / 'outputs' / 'metrics',
    'figures':  BASE / 'outputs' / 'figures',
    'tables':   BASE / 'outputs' / 'tables',
    'errors':   BASE / 'outputs' / 'error_analysis',
    'qlora':    BASE / 'outputs' / 'qlora',
    'pilot':    BASE / 'outputs' / 'pilot',
    'runs':     BASE / 'runs',
    'logs':     BASE / 'logs',
    'local':        LOCAL,
    'local_videos': LOCAL / 'videos',
    'local_repos':  LOCAL / 'repos',
    'local_tmp':    LOCAL / 'tmp',
}
for _p in P.values():
    _p.mkdir(parents=True, exist_ok=True)

In [3]:
# --- 3. Configuration ------------------------------------------------------------------
# config.yaml is written once from these defaults. Later edits to the YAML file win;
# keys added to the defaults in a newer version of this notebook are merged in without
# touching values that are already there.
DEFAULT_CONFIG = {
    'version': 1,
    'seed': 2026,
    'runtime': {
        'chunk_minutes': 110,          # stop a loop cleanly before the free-tier limits
        'progress_every': 25,
    },
    'data': {
        'max_questions_per_video': 4,  # spreads questions across videos (clustering)
        'dev_targets':  {'CW': 110, 'CH': 50, 'TN': 70, 'TC': 50, 'TP': 20},    # 300, NExT-QA val (NExT-GQA labelled)
        'grid_targets': {'CW': 220, 'CH': 100, 'TN': 140, 'TC': 100, 'TP': 40},  # 600, NExT-QA test (NExT-GQA labelled)
        'descriptive_targets': {'DO': 80, 'DL': 70, 'DC': 50},                  # 200, NExT-QA test
        'descriptive_max_per_video': 2,
        'intentqa_eval_size': 300,
        'robustness_size': 200,
        'pilot_videos': 10,
        'pilot_max_questions': 50,
        'anchor_agreement_tolerance_s': 1.0,
        'gap_short_max_s': 3.0,        # NExT-QA gap buckets use the ST-Gap thresholds
        'gap_long_min_s': 20.0,
    },
    'sources': {
        'nextgqa_repo': 'https://github.com/doc-doc/NExT-GQA.git',
        'intentqa_repo': 'https://github.com/JoseponLee/IntentQA.git',
        'nextqa_video_gdrive_id': '1jTcRCrVHS66ckOUfWRb-rXdzJ52XAWQH',
        'vidor_hf_repo': 'shangxd/vidor',
    },
    'cv': {
        'decode_fps': 5,
        'det_model': 'yolo26m.pt',          # exact weights confirmed in the pilot
        'det_conf': 0.25,
        'openvocab_model': 'yoloe-26m-seg.pt',
        'tracker': 'bytetrack.yaml',
        'min_track_frames_flag': 3,
        'stitch_max_gap_s': 2.0,
        'window_s': 3.0,
        'window_stride_s': 2.0,
        'frames_per_window': 4,
        'vlm_max_pixels': 448 * 448,
        'snap_tolerance_s': 1.0,
        'merge_min_tiou': 0.5,
    },
    'memory': {
        'overlap_tolerance_s': 0.5,
        'cooccur_window_s': 1.0,
    },
    'llm': {
        'retrieval_token_budget': 600,
        'expansion_max_items': 8,
        'max_new_tokens': 160,
        'temperature': 0.0,
        'self_consistency_samples': 3,
        'self_consistency_temperature': 0.7,
    },
    'models': {   # revisions are pinned to commit hashes in the week-2 pilot
        'reasoner_main':  {'id': 'Qwen/Qwen3-8B',              'revision': 'main', 'quant': 'nf4'},
        'reasoner_small': {'id': 'Qwen/Qwen3-4B',              'revision': 'main', 'quant': 'nf4'},
        'reasoner_alt':   {'id': 'google/gemma-3-4b-it',       'revision': 'main', 'quant': 'nf4', 'gated': True},
        'vlm_events':     {'id': 'Qwen/Qwen3-VL-4B-Instruct',  'revision': 'main', 'quant': 'fp16'},
        'vlm_baseline':   {'id': 'Qwen/Qwen3-VL-8B-Instruct',  'revision': 'main', 'quant': 'nf4'},
        'embedder':       {'id': 'BAAI/bge-small-en-v1.5',     'revision': 'main'},
        'crop_embedder':  {'id': 'google/siglip-base-patch16-224', 'revision': 'main'},
    },
}

import yaml

def _deep_merge(defaults, current):
    """Return current with any missing keys filled from defaults (current always wins)."""
    out = dict(current)
    for k, v in defaults.items():
        if k not in out:
            out[k] = v
        elif isinstance(v, dict) and isinstance(out[k], dict):
            out[k] = _deep_merge(v, out[k])
    return out

CONFIG_PATH = P['config'] / 'config.yaml'
if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        _loaded = yaml.safe_load(f) or {}
    CFG = _deep_merge(DEFAULT_CONFIG, _loaded)
    if CFG != _loaded:
        with open(CONFIG_PATH, 'w') as f:
            yaml.safe_dump(CFG, f, sort_keys=False)
        print('config.yaml: added new default keys (existing values kept)')
else:
    CFG = DEFAULT_CONFIG
    with open(CONFIG_PATH, 'w') as f:
        yaml.safe_dump(CFG, f, sort_keys=False)
    print(f'config.yaml created at {CONFIG_PATH}')

def config_hash(cfg=None):
    blob = json.dumps(cfg if cfg is not None else CFG, sort_keys=True).encode()
    return hashlib.sha1(blob).hexdigest()[:10]

print('config hash:', config_hash())

config.yaml created at /content/drive/MyDrive/ST-Reason/config/config.yaml
config hash: f5293d4309


In [4]:
# --- 4. Logging, seeds and safe file I/O --------------------------------------------------
SESSION_ID = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
LOG_PATH = P['logs'] / f'{ST_NOTEBOOK}_{SESSION_ID}.log'

def log(msg, level='INFO'):
    line = f'[{dt.datetime.now():%H:%M:%S}] {level:5s} {msg}'
    print(line)
    with open(LOG_PATH, 'a') as f:
        f.write(line + '\n')

def set_seed(seed=None):
    seed = CFG['seed'] if seed is None else seed
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if importlib.util.find_spec('numpy'):
        import numpy as np
        np.random.seed(seed)
    if 'torch' in sys.modules:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    return seed

def _json_default(o):
    if isinstance(o, Path):
        return str(o)
    if isinstance(o, set):
        return sorted(o)
    if hasattr(o, 'item'):            # numpy scalars
        return o.item()
    if hasattr(o, 'tolist'):          # numpy arrays
        return o.tolist()
    raise TypeError(f'Cannot serialise {type(o)}')

def save_json(obj, path):
    """Write JSON atomically: a crash mid-write never leaves a half-written file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=_json_default)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path) as f:
        return json.load(f)

def append_jsonl(path, record):
    """Append one record and flush it to disk immediately (one line per question)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'a') as f:
        f.write(json.dumps(record, ensure_ascii=False, default=_json_default) + '\n')
        f.flush(); os.fsync(f.fileno())

def read_jsonl(path):
    """Read a JSONL file, skipping a half-written last line left by a disconnect."""
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with open(path) as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                log(f'{path.name}: skipped unreadable line {i + 1} (likely an interrupted write)', 'WARN')
    return rows

def done_ids(target, key='uid'):
    """IDs already finished: JSON file stems in a folder, or `key` values in a JSONL file."""
    target = Path(target)
    if target.is_dir():
        return {p.stem for p in target.glob('*.json')}
    return {str(r[key]) for r in read_jsonl(target) if key in r}

def file_sha1(path, chunk=1 << 20):
    h = hashlib.sha1()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

def pip_install(*specs):
    """Install pinned packages quietly. Each notebook installs only what it needs."""
    if not specs:
        return
    log('pip install ' + ' '.join(specs))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *specs], check=True)

In [5]:
# --- 5. System report and memory monitoring --------------------------------------------------
def _gb(x):
    return round(x / 1024 ** 3, 2)

def gpu_info():
    try:
        out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                              '--format=csv,noheader,nounits'], capture_output=True, text=True, timeout=20)
        if out.returncode == 0 and out.stdout.strip():
            name, mem, drv = [s.strip() for s in out.stdout.strip().splitlines()[0].split(',')]
            return {'gpu': name, 'gpu_mem_gb': round(float(mem) / 1024, 1), 'driver': drv}
    except Exception:
        pass
    return {'gpu': None, 'gpu_mem_gb': 0, 'driver': None}

def sys_info():
    import psutil
    vm = psutil.virtual_memory()
    info = {
        'runtime': RUNTIME, 'python': platform.python_version(), 'cpus': os.cpu_count(),
        'ram_total_gb': _gb(vm.total), 'ram_available_gb': _gb(vm.available),
        'local_disk_free_gb': _gb(shutil.disk_usage(LOCAL).free),
        **gpu_info(),
    }
    for pkg in ['torch', 'transformers', 'ultralytics', 'bitsandbytes', 'pandas', 'numpy', 'cv2']:
        if importlib.util.find_spec(pkg):
            try:
                mod = sys.modules.get(pkg) or __import__(pkg)
                info[f'v_{pkg}'] = getattr(mod, '__version__', '?')
            except Exception as e:
                info[f'v_{pkg}'] = f'import failed: {e.__class__.__name__}'
    return info

class PeakMemory:
    """Context manager that records peak system RAM (sampled each second) and peak GPU memory."""
    def __init__(self, interval=1.0):
        self.interval, self.peak_ram_gb, self.peak_vram_gb = interval, 0.0, 0.0
    def _watch(self):
        import psutil
        proc = psutil.Process()
        while not self._stop.is_set():
            self.peak_ram_gb = max(self.peak_ram_gb, _gb(proc.memory_info().rss))
            time.sleep(self.interval)
    def __enter__(self):
        self._stop = threading.Event()
        self._t = threading.Thread(target=self._watch, daemon=True); self._t.start()
        if 'torch' in sys.modules:
            import torch
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
        return self
    def __exit__(self, *exc):
        self._stop.set(); self._t.join(timeout=5)
        if 'torch' in sys.modules:
            import torch
            if torch.cuda.is_available():
                self.peak_vram_gb = _gb(torch.cuda.max_memory_allocated())
        return False

def free_memory(*names):
    """Delete the named globals (e.g. 'model'), then clear Python and CUDA caches."""
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()
    if 'torch' in sys.modules:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def hf_login():
    """Read HF_TOKEN from Colab Secrets (key icon on the left) or the environment."""
    tok = os.environ.get('HF_TOKEN')
    if not tok and IN_COLAB:
        try:
            from google.colab import userdata
            tok = userdata.get('HF_TOKEN')
        except Exception:
            tok = None
    if tok:
        os.environ['HF_TOKEN'] = tok
        return True
    log('No HF_TOKEN found. Public models still work; Gemma needs a token and an accepted licence.', 'WARN')
    return False

In [6]:
# --- 6. Run manifest: every run leaves a record that paper numbers can be traced to ------------
MANIFEST_PATH = P['runs'] / 'manifest.csv'
MANIFEST_FIELDS = ['run_id', 'notebook', 'status', 'started', 'ended', 'minutes', 'items_done',
                   'items_total', 'runtime', 'gpu', 'peak_vram_gb', 'peak_ram_gb', 'config_hash',
                   'params', 'notes']

class Run:
    """Start a run, update it as work progresses, finish it with a status.
    A JSON file per run is written at start, so even a crashed run leaves a trace."""
    def __init__(self, notebook=None, params=None, items_total=None):
        self.notebook = notebook or ST_NOTEBOOK
        self.params = params or {}
        stamp = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
        tag = hashlib.sha1(json.dumps(self.params, sort_keys=True, default=str).encode()).hexdigest()[:6]
        self.run_id = f'{self.notebook}-{stamp}-{tag}'
        self.t0 = time.time()
        self.mem = PeakMemory().__enter__()
        self.state = {'run_id': self.run_id, 'notebook': self.notebook, 'status': 'running',
                      'started': dt.datetime.now().isoformat(timespec='seconds'),
                      'items_done': 0, 'items_total': items_total, 'runtime': RUNTIME,
                      'gpu': gpu_info()['gpu'], 'config_hash': config_hash(), 'params': self.params,
                      'sys': sys_info()}
        save_json(self.state, P['runs'] / f'{self.run_id}.json')
        log(f'run started: {self.run_id}')

    def update(self, **kw):
        self.state.update(kw)
        save_json(self.state, P['runs'] / f'{self.run_id}.json')

    def finish(self, status='done', notes=''):
        self.mem.__exit__(None, None, None)
        self.state.update({'status': status, 'notes': notes,
                           'ended': dt.datetime.now().isoformat(timespec='seconds'),
                           'minutes': round((time.time() - self.t0) / 60, 1),
                           'peak_ram_gb': self.mem.peak_ram_gb, 'peak_vram_gb': self.mem.peak_vram_gb})
        save_json(self.state, P['runs'] / f'{self.run_id}.json')
        new = not MANIFEST_PATH.exists()
        with open(MANIFEST_PATH, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS, extrasaction='ignore')
            if new:
                w.writeheader()
            row = dict(self.state); row['params'] = json.dumps(self.params, default=str)
            w.writerow(row)
        log(f"run finished: {self.run_id} [{status}] {self.state['minutes']} min, "
            f"peak RAM {self.mem.peak_ram_gb} GB, peak VRAM {self.mem.peak_vram_gb} GB")

In [7]:
# --- 7. Resumable processing loop ------------------------------------------------------------
def run_resumable(items, key, fn, done=None, budget_min=None, progress_every=None, desc='items'):
    """Process `items` in order, skipping keys already in `done`.

    fn(item) must save its own result to Drive (one file or one JSONL line per item).
    The loop stops cleanly when the time budget is used, so a free-tier session never
    ends mid-write. Re-running the cell continues where it stopped.
    Returns a dict with counts; failures are logged and skipped, not fatal.
    """
    done = set(done or ())
    budget_s = 60 * (budget_min if budget_min is not None else CFG['runtime']['chunk_minutes'])
    every = progress_every or CFG['runtime']['progress_every']
    todo = [it for it in items if str(key(it)) not in done]
    log(f'{desc}: {len(items)} total, {len(items) - len(todo)} already done, {len(todo)} to do')
    t0, n_ok, n_fail = time.time(), 0, 0
    for i, it in enumerate(todo, 1):
        if time.time() - t0 > budget_s:
            log(f'time budget reached after {n_ok} items; re-run this cell to continue', 'WARN')
            break
        try:
            fn(it)
            n_ok += 1
        except KeyboardInterrupt:
            raise
        except Exception as e:
            n_fail += 1
            log(f'{desc} {key(it)} failed: {e.__class__.__name__}: {e}', 'ERROR')
        if i % every == 0:
            rate = (time.time() - t0) / i
            log(f'{desc}: {i}/{len(todo)} | {rate:.1f}s each | ETA {rate * (len(todo) - i) / 60:.1f} min')
    remaining = len(todo) - n_ok - n_fail
    return {'total': len(items), 'skipped': len(items) - len(todo), 'ok': n_ok, 'failed': n_fail,
            'remaining': remaining, 'seconds': round(time.time() - t0, 1)}

In [8]:
# --- 8. Videos: stage zips to the fast local disk, and probe files -----------------------------
def stage_zip_to_local(zip_path, dest_dir=None):
    """Copy a video zip from Drive to the Colab disk and unzip it (skipping files already there).
    Reading many small files through the Drive mount is slow; local reads are fast."""
    zip_path = Path(zip_path)
    dest_dir = Path(dest_dir or P['local_videos'])
    dest_dir.mkdir(parents=True, exist_ok=True)
    local_zip = P['local_tmp'] / zip_path.name
    if not local_zip.exists() or local_zip.stat().st_size != zip_path.stat().st_size:
        log(f'copying {zip_path.name} ({_gb(zip_path.stat().st_size)} GB) to local disk')
        shutil.copy2(zip_path, local_zip)
    n_new = 0
    with zipfile.ZipFile(local_zip) as zf:
        for m in zf.infolist():
            if m.is_dir():
                continue
            out = dest_dir / m.filename
            if not out.exists() or out.stat().st_size != m.file_size:
                zf.extract(m, dest_dir); n_new += 1
    log(f'{zip_path.name}: {n_new} files extracted to {dest_dir}')
    return dest_dir

def probe_video(path):
    """Frame count, fps, size and duration via OpenCV; ok=False if the first frame cannot be read."""
    import cv2
    cap = cv2.VideoCapture(str(path))
    try:
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
        fps = float(cap.get(cv2.CAP_PROP_FPS) or 0)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
        ok, _ = cap.read()
    finally:
        cap.release()
    return {'frames': n, 'fps': round(fps, 3), 'width': w, 'height': h,
            'duration_s': round(n / fps, 2) if fps else None, 'ok': bool(ok and n > 0 and fps > 0)}

In [9]:
# --- 9. Backup platform (Kaggle): move outputs into Drive ------------------------------------
def pack_outputs(subdirs=('outputs', 'runs', 'logs'), name=None):
    """On Kaggle: zip the given project subfolders. Download the zip and upload it to
    MyDrive/ST-Reason/inbox/, then run merge_inbox() in Colab."""
    name = name or f'{RUNTIME}_{SESSION_ID}.zip'
    out = BASE.parent / name
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
        for sd in subdirs:
            for f in (BASE / sd).rglob('*'):
                if f.is_file():
                    zf.write(f, f.relative_to(BASE))
    log(f'packed {out} ({_gb(out.stat().st_size)} GB)')
    return out

def merge_inbox():
    """In Colab: unpack zips from inbox/ into the project. Existing files are never overwritten,
    except runs/manifest.csv, whose rows are appended."""
    for z in sorted(P['inbox'].glob('*.zip')):
        added, kept = 0, 0
        with zipfile.ZipFile(z) as zf:
            for m in zf.infolist():
                if m.is_dir():
                    continue
                target = BASE / m.filename
                if m.filename == 'runs/manifest.csv' and target.exists():
                    lines = zf.read(m).decode().splitlines()[1:]
                    with open(target, 'a') as f:
                        f.write('\n'.join(lines) + ('\n' if lines else ''))
                    continue
                if target.exists():
                    kept += 1; continue
                target.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
                added += 1
        z.rename(z.with_suffix('.merged'))
        log(f'{z.name}: {added} files added, {kept} already present (kept)')

In [11]:
# --- 10. Self-test (only when this notebook is run on its own) ---------------------------------
if not globals().get('ST_QUIET', False):
    set_seed()
    info = sys_info()
    for k, v in info.items():
        print(f'{k:22s} {v}')
    if info['ram_total_gb'] and info['ram_total_gb'] < 14:
        print('\nNote: system RAM is about', info['ram_total_gb'], 'GB. Notebooks stream frames and free memory between stages.')

    # check the helpers round-trip correctly in the local scratch folder
    t = P['local_tmp'] / 'selftest.jsonl'
    t.unlink(missing_ok=True)
    for i in range(3):
        append_jsonl(t, {'uid': f'q{i}', 'ok': True})
    with open(t, 'a') as f:
        f.write('{"uid": "half-writ')            # simulate a disconnect mid-write
    assert done_ids(t) == {'q0', 'q1', 'q2'}, 'done_ids failed'
    save_json({'a': 1}, P['local_tmp'] / 'selftest.json')
    assert load_json(P['local_tmp'] / 'selftest.json') == {'a': 1}
    res = run_resumable(list(range(5)), key=str, fn=lambda x: None, done={'0', '1'}, desc='selftest')
    assert res['ok'] == 3 and res['skipped'] == 2
    print('\nHelpers OK.')
    hf_login()
    print('\nProject folders:')
    for k in ['config', 'data', 'ann', 'subsets', 'source', 'stgap', 'outputs', 'runs', 'logs']:
        print(f'  {k:8s} {P[k]}')

runtime                colab
python                 3.13.16
cpus                   2
ram_total_gb           12.67
ram_available_gb       11.25
local_disk_free_gb     87.19
gpu                    None
gpu_mem_gb             0
driver                 None
v_torch                2.11.0+cpu
v_transformers         5.18.0
v_pandas               2.2.3
v_numpy                2.1.3
v_cv2                  5.0.0

Note: system RAM is about 12.67 GB. Notebooks stream frames and free memory between stages.
[09:47:38] WARN  selftest.jsonl: skipped unreadable line 4 (likely an interrupted write)
[09:47:38] INFO  selftest: 5 total, 2 already done, 3 to do

Helpers OK.

Project folders:
  config   /content/drive/MyDrive/ST-Reason/config
  data     /content/drive/MyDrive/ST-Reason/data
  ann      /content/drive/MyDrive/ST-Reason/data/annotations
  subsets  /content/drive/MyDrive/ST-Reason/data/subsets
  source   /content/drive/MyDrive/ST-Reason/data/source
  stgap    /content/drive/MyDrive/ST-Reason/data/

## One-time setup checklist

1. **Copy the notebooks into Drive.** Put all `.ipynb` files in `MyDrive/ST-Reason/notebooks/`; the others load this one from there.
2. **Hugging Face token** (needed for Gemma 3 in Tier 2; harmless otherwise). Create a read token at huggingface.co → Settings → Access Tokens, accept the Gemma 3 licence on its model page, then in Colab open the key icon (Secrets) and add `HF_TOKEN`, with notebook access on.
3. **Team accounts.** Share the `ST-Reason` folder with each member as Editor, and have each member add a shortcut to it in their own `MyDrive`, so the same paths work for everyone.
4. **Version control (optional).** In Colab, File → Save a copy in GitHub keeps notebook versions in your repo; data and results stay in Drive.